In [ ]:
import redis
import mysql.connector
import json
import hashlib
import pwinput
# 1. Connect to Redis on Ubuntu localhost
r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)

# 2. Connect to MySQL
db = mysql.connector.connect(
    host=input ("your localhost"),
    user=input ("your_mysql_user"),
    password=pwinput.pwinput ("your_mysql_password"),
    database=input ("your_database")    
)

def get_user_profile(user_id):
    cache_key = f"user:{user_id}"
    
    # Step A: Check Redis cache first
    cached_data = r.get(cache_key)
    if cached_data:
        print("-> Serving from Redis Cache (Hit)")
        return json.loads(cached_data)
    
    print("-> Cache Miss. Querying MySQL...")
print("Validation successful")

In [ ]:
def get_user_profile(user_id):
    cache_key = f"user:{user_id}"
    
    # Step A: Check Redis cache first
    cached_data = r.get(cache_key)
    if cached_data:
        print("-> Serving from Redis Cache (Hit)")
        return json.loads(cached_data)
    
    print("-> Cache Miss. Querying MySQL...")
    
    # Step B: Query MySQL on cache miss
    cursor = db.cursor(dictionary=True)
    cursor.execute("SELECT id, name, email FROM users WHERE id = %s", (user_id,))
    user = cursor.fetchone()
    cursor.close()
    
    if user:
        # Step C: Store result in Redis with a TTL (e.g., 10 minutes / 600 seconds)
        r.setex(cache_key, 600, json.dumps(user))
        
    return user

def update_user_email(user_id, new_email):
    # Step A: Write to MySQL system of record
    cursor = db.cursor()
    cursor.execute("UPDATE users SET email = %s WHERE id = %s", (new_email, user_id))
    db.commit()
    cursor.close()
    
    # Step B: Invalidate/Delete the cache to force fresh fetch on next read
    cache_key = f"user:{user_id}"
    r.delete(cache_key)
    print("-> Database updated and Redis cache invalidated.")